# UFC Analytics Platform — Project Writeup

**Goal:** Build a production-grade ML system to predict UFC fight outcomes using only information available before each fight (strictly no data leakage).

**Stack:** Python · PostgreSQL (Supabase) · FastAPI · scikit-learn · XGBoost · SHAP

**Reference methodology:** Turgut (2021), Tilburg University — *"Machine Learning approach to predicting MMA matches"*

---

## Pipeline Overview

```
UFCStats.com scraping
      ↓
PostgreSQL (Supabase) — 756 events, 4,449 fighters, 8,482 fights
      ↓
ETL pipeline (4 phases) — FK resolution, type parsing, derived columns
      ↓
Feature engineering (7 modules) — 71 candidate features
      ↓
Feature selection (MI + collinearity) — 31 features selected
      ↓
Temporal split 70 / 15 / 15
      ↓
Three candidate models — LR, XGBoost, Random Forest
      ↓
Best model: Random Forest  test_acc=63.85%  test_AUC=0.6785
      ↓
SHAP interpretation
```

In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import Image, display

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Paths (notebook lives in docs/, data lives in backend/)
ROOT     = Path('..').resolve()
BACKEND  = ROOT / 'backend'
FEATURES = BACKEND / 'features'
MODELS   = BACKEND / 'models'
EDA      = ROOT / 'eda'

print('Root:', ROOT)
print('Parquet:', (FEATURES / 'training_data.parquet').exists())
print('Models dir:', MODELS.exists())

---
## Phase 1: Data Collection

**Source:** [UFCStats.com](http://ufcstats.com) — the official statistical record for every UFC event since 1994.

**Scraper:** A custom BeautifulSoup scraper (`live_scraper.py`) writes to six PostgreSQL tables hosted on Supabase. A GitHub Actions workflow runs it weekly after each UFC event.

### Database Schema

| Table | Rows | Description |
|---|---|---|
| `event_details` | 756 | UFC events with date and location |
| `fighter_details` | 4,449 | Fighter profiles (name, URL) |
| `fighter_tott` | 4,435 | Tale of the Tape: height, weight, reach, DOB, stance |
| `fight_details` | 8,482 | Fight matchups with FK to event |
| `fight_results` | 8,482 | Outcome, method, round, time |
| `fight_stats` | 39,912 | Per-fighter, per-round statistics |

All six tables are FK-linked with 99.75%+ coverage across all relationships.

**Date range:** 1994-03-11 (UFC 1) to 2025-12-07 (Fight Night: Covington vs. Buckley)

In [ ]:
# Load the training parquet — every row is one fight from one fighter's perspective
df = pd.read_parquet(FEATURES / 'training_data.parquet')
df['event_date'] = pd.to_datetime(df['event_date'])

print(f"Training matrix shape: {df.shape}")
print(f"Date range: {df['event_date'].min().date()} → {df['event_date'].max().date()}")
print(f"\nColumns: {list(df.columns)}")

In [ ]:
# Fights per year (each row in the parquet = 1 fight, deduplicated)
fights_by_year = (
    df.drop_duplicates(subset='fight_id')
      .set_index('event_date')
      .resample('YE')
      .size()
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(fights_by_year.index.year, fights_by_year.values, color='#1565C0', alpha=0.85, width=0.75)
ax.set_xlabel('Year')
ax.set_ylabel('Fights in training set')
ax.set_title('UFC Fights Per Year (post-1998 cutoff, debuts and SHW removed)')
ax.yaxis.set_major_locator(mticker.MultipleLocator(50))
plt.tight_layout()
plt.show()

print(f"Total unique fights in training set: {df['fight_id'].nunique()}")

---
## Phase 2: ETL Pipeline

Raw scraped data has several data quality problems that must be resolved before any feature engineering.

### Four-phase ETL (`post_scrape_clean.py`)

| Phase | Script logic | Example |
|---|---|---|
| **1. FK Resolution** | Match fighter names to IDs via URL; resolve `fighter_a_id`, `winner_id` | `"Conor McGregor"` → `fighter_id=abc123` |
| **2. Quality Cleanup** | Replace `"--"` with NULL; strip trailing spaces from METHOD | `"--"` → `NULL` |
| **3. Type Parsing** | Parse `"17 of 37"` → `sig_str_landed=17, sig_str_attempted=37`; `"1:23"` → `ctrl_seconds=83` | Numeric typed columns |
| **4. Derived Columns** | Infer `weight_class` from BOUT string; set `is_title_fight`, `is_championship_rounds` | `"UFC Heavyweight Championship"` → flag |

A standalone `validate_etl.py` script runs after each ETL pass and exits with code 1 if data quality thresholds are not met. This is integrated into the GitHub Actions post-scrape workflow.

### Three Data Quality Filters (applied in `pipeline.py` before feature engineering)

Following Tilburg (2021):

| Filter | Rows removed | Reason |
|---|---|---|
| Pre-October 1998 cutoff | 169 | Frequent missing stats in early UFC |
| Debut fights | 2,009 | Either fighter has zero prior UFC fights; all stat features are zero — pure noise |
| Super Heavyweight / Open Weight | 1 | No weight limit; incompatible with class-based features |

In [ ]:
# Weight class distribution in the training set
wc_counts = (
    df.drop_duplicates(subset='fight_id')['weight_class']
      .value_counts()
      .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(wc_counts.index, wc_counts.values, color='#1565C0', alpha=0.8)
ax.set_xlabel('Number of fights')
ax.set_title('Fight Count by Weight Class (training set)')
plt.tight_layout()
plt.show()

In [ ]:
# Target balance — should be ~50/50 after random perspective swap
target_counts = df['fighter_a_wins'].value_counts()
pct = target_counts / len(df) * 100

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Fighter B wins (0)', 'Fighter A wins (1)'], target_counts.values, 
       color=['#E53935', '#1565C0'], alpha=0.85)
for i, (v, p) in enumerate(zip(target_counts.values, pct.values)):
    ax.text(i, v + 20, f'{p:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Count')
ax.set_title('Target Class Balance After Perspective Swap')
ax.set_ylim(0, max(target_counts.values) * 1.12)
plt.tight_layout()
plt.show()

print(f"Class balance: {pct.to_dict()}")

---
## Phase 3: Feature Engineering

All features are **point-in-time correct** — every value at fight `i` reflects only fights `0 … i-1`. This is verified by the `shift(1)` pattern used throughout all modules.

### Seven Feature Modules

| Module | Output | Key technique |
|---|---|---|
| `differentials.py` | Win rate, win/loss streaks, experience, opponent quality | `cumsum().shift(1)` |
| `rolling_metrics.py` | 3/5/7-fight rolling avg + career avg per stat | `rolling(N).mean().shift(1)`, `expanding().mean().shift(1)` |
| `style_features.py` | Grappling ratio, aggression score, KO rate, decision rate, defense score | `cumsum().shift(1)` per-fight |
| `time_features.py` | Age at fight, career length, days in weight class, days since last fight | Date arithmetic |
| `opponent_quality.py` | Avg opponent losses, strength of schedule | Point-in-time opponent record lookup |
| `differentials.py` | Physical diffs: reach, weight | From `fighter_tott` table |
| `pipeline.py` | Assembles all modules, applies perspective swap | Negates all `diff_*` columns for swapped rows |

### Leakage Prevention

Every rolling and cumulative operation uses `.shift(1)` to exclude the current fight from its own features. The critical invariant:

> **Value at row i = f(rows 0 … i-1)** — the current fight never contributes to its own features.

This is the same standard Tilburg (2021) enforces. Without it, accuracy inflates by 6–10% (Tilburg's controlled leakage experiment).

### Career Averages vs Rolling Windows

We include **both** approaches:
- **Rolling windows (3/5/7):** Capture recent form — useful when a fighter is on a hot streak or has been inactive
- **Career averages (expanding mean):** Tilburg's approach — captures long-run baseline; more stable, less sensitive to short-run variance

Feature selection then determines which windows are most predictive for each stat.

In [ ]:
with open(FEATURES / 'selected_features.json') as f:
    sel = json.load(f)

feature_names = sel['feature_names']
print(f"Selected features ({len(feature_names)}): {feature_names}")

In [ ]:
# Distribution of the top 9 numeric features
top9 = feature_names[:9]

fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, col in zip(axes.flat, top9):
    data = df[col].dropna()
    ax.hist(data, bins=50, color='#1565C0', alpha=0.75, edgecolor='none')
    ax.set_title(col, fontsize=8)
    ax.set_xlabel('Value', fontsize=7)
    ax.set_ylabel('Count', fontsize=7)
    ax.tick_params(labelsize=7)

fig.suptitle('Distribution of Top 9 Features (training set)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### Correlation Heatmap (from EDA phase)

The heatmap below was produced during exploratory analysis and shows the collinearity structure of the full feature candidate pool before selection. Pairs with |r| > 0.90 are identified for pruning.

In [ ]:
corr_img = EDA / 'correlation_heatmap_enhanced.png'
if corr_img.exists():
    display(Image(filename=str(corr_img), width=900))
else:
    print(f'Image not found: {corr_img}')

---
## Phase 4: Feature Selection

Starting from 71 candidate features, two filters are applied in sequence:

### Step 1: Collinearity Pruning (|r| > 0.90)

Pairs of features with Pearson correlation above 0.90 are identified. For each pair, the feature with the lower mutual information (MI) score is dropped. This removes redundant features that would add noise without adding information.

**32 features removed** at this step.

### Step 2: Zero-MI Filter

Features with MI score = 0 (no measurable association with the target) are removed.

**8 features removed** at this step.

### Result: 31 features selected

MI (mutual information) measures statistical dependence without assuming linearity, making it appropriate for a dataset where many relationships are non-linear (e.g., win streaks, grappling ratios).

In [ ]:
mi_scores = sel['mi_scores']
mi_df = pd.Series(mi_scores).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))
colors = ['#1565C0' if v > 0.005 else '#78909C' for v in mi_df.values]
ax.barh(range(len(mi_df)), mi_df.values, color=colors, alpha=0.85)
ax.set_yticks(range(len(mi_df)))
ax.set_yticklabels(mi_df.index, fontsize=8)
ax.set_xlabel('Mutual Information Score', fontsize=11)
ax.set_title('Feature Mutual Information Scores\n(blue = MI > 0.005, grey = weaker signal)', fontsize=12)
ax.axvline(0.005, color='red', linestyle='--', alpha=0.5, linewidth=1)
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 removed collinear pairs
removed = sel['removed_collinear']
removed_df = pd.DataFrame(removed).sort_values('r', ascending=False)
print('Top 10 removed collinear pairs:')
removed_df[['feature', 'correlated_with', 'r', 'mi_score_dropped', 'mi_score_kept']].head(10).to_string(index=False)

In [ ]:
# Correlation distribution of removed pairs
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist([p['r'] for p in removed], bins=15, color='#E53935', alpha=0.75, edgecolor='white')
ax.set_xlabel('Pearson |r|')
ax.set_ylabel('Count')
ax.set_title('Distribution of Correlation Coefficients for Removed Feature Pairs')
ax.axvline(0.90, color='black', linestyle='--', label='Threshold (0.90)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Phase 5: Model Training

### Data Split Strategy

Matches are **never shuffled** — they are kept in chronological order (oldest first) to avoid training on future fights relative to validation/test data. This follows Tilburg (2021) and Tax & Joustra (2015).

| Split | Proportion | Approx rows | Date range |
|---|---|---|---|
| Train | 70% | ~4,361 | 1998 → 2021 |
| Validation | 15% | ~935 | 2021 → 2023 |
| Test | 15% | ~935 | 2023 → 2025 |

**Model selection uses validation ROC-AUC only.** The test set is touched exactly once, at the end.

### Three Candidate Models

| Model | Key regularisation | Notes |
|---|---|---|
| Logistic Regression | C=0.1, balanced class weights | Interpretable linear baseline |
| XGBoost | max_depth=3, early stopping (50 rounds), reg_α=1, reg_λ=2 | Gradient boosting; early stopping on val set |
| Random Forest | max_depth=6, min_samples_leaf=30, balanced weights | Best performer; winner |

### Preprocessing Pipeline (inside sklearn Pipeline)

- Numeric features: median imputation → StandardScaler
- Categorical (`weight_class`): most-frequent imputation → OrdinalEncoder (lightest → heaviest)

In [ ]:
# Load the most recent experiment log entry
log_path = MODELS / 'experiment_log.json'
if log_path.exists():
    with open(log_path) as f:
        experiment_log = json.load(f)
    latest = experiment_log[-1]
    print(f"Run timestamp: {latest['timestamp']}")
    print(f"Features: {latest['n_features']}  |  Train rows: {latest['train_rows']}  |  Test rows: {latest['test_rows']}")
    print(f"Best model: {latest['best_model']}")
    models_results = latest['models']
else:
    print('experiment_log.json not found — run python -m ml.run_train first')
    models_results = {}

In [ ]:
if models_results:
    metrics_df = pd.DataFrame(models_results).T
    print('\nAll model metrics:')
    print(metrics_df.to_string())

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    model_names = [n.replace('_', '\n') for n in metrics_df.index]
    colors = ['#78909C', '#1E88E5', '#1565C0']

    for ax, col, title in zip(
        axes,
        ['val_auc', 'test_accuracy', 'test_auc'],
        ['Validation AUC\n(model selection)', 'Test Accuracy', 'Test AUC'],
    ):
        vals = metrics_df[col].astype(float)
        bars = ax.bar(model_names, vals, color=colors, alpha=0.85)
        best_idx = vals.argmax()
        bars[best_idx].set_edgecolor('gold')
        bars[best_idx].set_linewidth(2.5)
        ax.set_ylim(vals.min() * 0.97, vals.max() * 1.03)
        ax.set_title(title, fontsize=11)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                    f'{v:.4f}', ha='center', va='bottom', fontsize=9)

    fig.suptitle('Model Comparison — Temporal Test Set (gold outline = best)', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# Compare to Tilburg (2021) baselines
comparison = pd.DataFrame({
    'Study': ['Tilburg RF (2021)', 'Tilburg ANN (2021)', 'This project RF', 'This project XGBoost', 'This project LR'],
    'Test Accuracy': [0.5898, 0.5911, None, None, None],
    'Leakage-free': ['Yes', 'Yes', 'Yes', 'Yes', 'Yes'],
    'Features': [34, 34, 32, 32, 32],
    'Training rows': [3925, 3925, None, None, None],
})

if models_results:
    for i, name in enumerate(['random_forest', 'xgboost', 'logistic_regression']):
        if name in models_results:
            comparison.loc[comparison['Study'].str.contains(name.replace('_', ' ').split()[0].title()), 'Test Accuracy'] = models_results[name]['test_accuracy']
            comparison.loc[comparison['Study'].str.contains(name.replace('_', ' ').split()[0].title()), 'Training rows'] = latest['train_rows']

print(comparison.to_string(index=False))

In [ ]:
# RF feature importance (impurity-based — from the trained model)
fi_path = MODELS / 'feature_importance.json'
if fi_path.exists():
    with open(fi_path) as f:
        fi = json.load(f)
    fi_series = pd.Series(fi).sort_values(ascending=True).tail(20)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(range(len(fi_series)), fi_series.values, color='#1565C0', alpha=0.8)
    ax.set_yticks(range(len(fi_series)))
    ax.set_yticklabels(fi_series.index, fontsize=9)
    ax.set_xlabel('Gini Impurity Importance', fontsize=11)
    ax.set_title('Random Forest Feature Importance — Top 20 (impurity-based)', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('feature_importance.json not found')

---
## Phase 6: SHAP Analysis

SHAP (SHapley Additive exPlanations) provides model-agnostic feature attribution grounded in cooperative game theory. For each prediction, SHAP assigns each feature a value representing its contribution to the output relative to the model's expected output.

**Why SHAP over RF feature importance?**

| Method | Captures non-linearity | Per-prediction | Bias toward high-cardinality | Direction of effect |
|---|---|---|---|---|
| RF feature importance (impurity) | Partial | No (global only) | Yes | No |
| MI score | Yes | No | No | No |
| **SHAP** | **Yes** | **Yes** | **No** | **Yes** |

**Mean |SHAP|** is the global importance metric used here — it answers *"how much did this feature move predictions on average, in absolute terms?"* Positive SHAP = pushes toward Fighter A winning. Negative SHAP = pushes toward Fighter B winning.

Run `python -m ml.run_train` to generate `models/shap_summary.json` and `models/shap_importance.png`.

In [ ]:
shap_path = MODELS / 'shap_summary.json'
if shap_path.exists():
    with open(shap_path) as f:
        shap_summary = json.load(f)

    shap_series = pd.Series(shap_summary).sort_values(ascending=True).tail(20)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(range(len(shap_series)), shap_series.values, color='#2196F3', alpha=0.85)
    ax.set_yticks(range(len(shap_series)))
    ax.set_yticklabels(shap_series.index, fontsize=9)
    ax.set_xlabel('Mean |SHAP value|', fontsize=11)
    ax.set_title('SHAP Feature Importance — Top 20 (test set attribution)', fontsize=12)
    plt.tight_layout()
    plt.show()

    print('\nTop 10 features by SHAP:')
    for i, (feat, val) in enumerate(list(shap_summary.items())[:10], 1):
        print(f'  {i:2d}. {feat:<40s} {val:.5f}')
else:
    print('shap_summary.json not found.')
    print('Run: cd backend && python -m ml.run_train')
    print('Then re-run this cell.')

In [ ]:
# Compare MI ranking vs SHAP ranking for the same features
if shap_path.exists():
    mi_rank  = {f: i+1 for i, f in enumerate(pd.Series(mi_scores).sort_values(ascending=False).index)}
    shap_rank = {f: i+1 for i, f in enumerate(shap_summary.keys())}

    common = [f for f in feature_names if f in shap_rank]
    rank_df = pd.DataFrame({
        'Feature': common,
        'MI rank': [mi_rank.get(f, '—') for f in common],
        'SHAP rank': [shap_rank.get(f, '—') for f in common],
    }).sort_values('SHAP rank')
    rank_df['Rank shift'] = rank_df['MI rank'].astype(int) - rank_df['SHAP rank'].astype(int)

    print('MI rank vs SHAP rank comparison:')
    print(rank_df.to_string(index=False))

In [ ]:
# Display the saved SHAP chart generated by train.py
shap_png = MODELS / 'shap_importance.png'
if shap_png.exists():
    display(Image(filename=str(shap_png), width=750))
else:
    print('shap_importance.png not found — run python -m ml.run_train')

---
## Appendix A: Prior EDA — DeepUFC Replication

Before building the full pipeline, the DeepUFC (2017) paper was replicated to establish a baseline and verify that the data setup was correct.

**DeepUFC approach:** 9 differential features from current career stats (no rolling windows, no leakage prevention). Reported 72% accuracy on 2017 data.

**Result on 2025 dataset:** 84.69% accuracy — an apparent improvement. However, this model uses **current** career summary stats (post-hoc summaries containing future fight data relative to each prediction point). This is the information leakage Tilburg (2021) documents — the realistic leakage-free ceiling is ~59–64%.

The DeepUFC replication confirms that leakage-inflated accuracy (~65–85%) is achievable but meaningless for predicting future fights.

In [ ]:
for img_name, caption in [
    ('deepufc_training_history.png', 'DeepUFC Training History (loss & accuracy curves)'),
    ('deepufc_confusion_matrix.png', 'DeepUFC Confusion Matrix'),
    ('deepufc_feature_importance.png', 'DeepUFC Feature Importance'),
]:
    p = EDA / img_name
    if p.exists():
        print(f'\n{caption}')
        display(Image(filename=str(p), width=600))
    else:
        print(f'Not found: {p}')

---
## Phase 8: Conclusions

### Results Summary

| Metric | Value |
|---|---|
| Win/loss test accuracy | **63.85%** |
| Win/loss test AUC | **0.6785** |
| Method prediction accuracy | ~60% |
| Training rows (post-filter) | 6,230 |
| Features selected | 31 |
| Best model | Random Forest |
| Val–test gap | −0.04 (no overfitting) |

### Why 63.85% vs Tilburg's 59%?

The gap is genuine (not leakage) and comes from three sources:

1. **Rolling windows** capture recent form rather than just career averages — a fighter on a 7-fight win streak is better represented by their last 7 fights than by a decade of accumulated data
2. **Style features** (grappling ratio, aggression score, KO rate, decision rate, defense score) are entirely absent from Tilburg's feature set and add real signal
3. **60% more training data** — 6,230 rows vs 3,925, due to 5 additional years of UFC history

Tilburg's 59% is the ceiling for *his specific feature set*, not for MMA prediction in general.

### Remaining Gaps vs Tilburg

| Gap | Impact |
|---|---|
| Per-minute normalization of stats | Medium — early finishers have lower raw totals; we compensate with multiple window sizes |
| Strike zone features (head/body/leg/distance/clinch/ground) | Medium — 12 Tilburg features we do not compute |
| Submission attempts, reversals | Low — rarely decisive in predictions |
| MICE imputation for height/reach | Low — median vs MICE, functionally similar for sparse missingness |

### Next Steps

1. **Strike zone features** — add head/body/leg/distance/clinch/ground differentials from `fight_stats` 
2. **Per-minute normalization** — join fight duration from `fight_results.total_fight_time_seconds` and divide all rolling stats by minutes
3. **RandomizedSearchCV** — hyperparameter search (Tilburg used this; we used manual tuning)
4. **Frontend** — React dashboard with real-time prediction sliders (Task 7)
5. **Style evolution dashboard** — timeline of finish rates by era/weight class (Task 8)